Check Null Values

In [0]:
SELECT 
  'DQ_CHECK_1_NO_NULL_PK' as check_name,
  CASE 
    WHEN COUNT(*) = 0 THEN 'PASS' 
    ELSE 'FAIL' 
  END as status,
  COUNT(*) as null_count
FROM nestle_dev_bronze.sql_server.sales_transactions
WHERE transaction_id IS NULL 
   OR product_id IS NULL 
   OR region IS NULL;

Check Amount Positive

In [0]:
SELECT 
  'DQ_CHECK_2_AMOUNT_POSITIVE' as check_name,
  CASE 
    WHEN COUNT(*) = 0 THEN 'PASS' 
    ELSE 'FAIL' 
  END as status,
  COUNT(*) as invalid_amount_count
FROM nestle_dev_bronze.sql_server.sales_transactions
WHERE amount <= 0;

Check Valid Channels

In [0]:
SELECT 
  'DQ_CHECK_3_VALID_CHANNELS' as check_name,
  CASE 
    WHEN COUNT(*) = 0 THEN 'PASS' 
    ELSE 'FAIL' 
  END as status,
  COUNT(*) as invalid_channel_count
FROM nestle_dev_bronze.sql_server.sales_transactions
WHERE channel NOT IN ('Online', 'Retail', 'B2B', 'Wholesale');

Summary - Total Rows

In [0]:
SELECT 
  COUNT(*) as total_rows,
  COUNT(DISTINCT transaction_id) as unique_transactions,
  MIN(DATE(created_at)) as earliest_date,
  MAX(DATE(created_at)) as latest_date
FROM nestle_dev_bronze.sql_server.sales_transactions;

## Data Fix — Reset Bronze Table to 400 Rows
Truncate and reload from all 3 CSVs to eliminate accumulated duplicate loads.

In [0]:
TRUNCATE TABLE nestle_dev_bronze.sql_server.sales_transactions

In [0]:
-- Day 1: full load (no watermark filter)
INSERT INTO nestle_dev_bronze.sql_server.sales_transactions
  (transaction_id, product_id, region, channel, customer_id, quantity, unit_price, amount, created_at, modified_at, ingestion_timestamp)
SELECT
  CAST(transaction_id AS STRING), product_id, region, channel, customer_id,
  CAST(quantity AS BIGINT), unit_price, amount,
  CAST(created_at AS STRING), CAST(modified_at AS STRING), current_timestamp()
FROM read_files('/Volumes/nestle_dev_bronze/data_lake/landing/sql_sales_transactions_day1.csv', format => 'csv', header => true)
UNION ALL
-- Day 2: modified_at after Day 1 watermark (2026-06-20 23:59:59)
SELECT
  CAST(transaction_id AS STRING), product_id, region, channel, customer_id,
  CAST(quantity AS BIGINT), unit_price, amount,
  CAST(created_at AS STRING), CAST(modified_at AS STRING), current_timestamp()
FROM read_files('/Volumes/nestle_dev_bronze/data_lake/landing/sql_sales_transactions_day2.csv', format => 'csv', header => true)
WHERE modified_at > '2026-06-20 23:59:59'
UNION ALL
-- Day 3: modified_at after Day 2 watermark (2026-06-21 23:59:59)
SELECT
  CAST(transaction_id AS STRING), product_id, region, channel, customer_id,
  CAST(quantity AS BIGINT), unit_price, amount,
  CAST(created_at AS STRING), CAST(modified_at AS STRING), current_timestamp()
FROM read_files('/Volumes/nestle_dev_bronze/data_lake/landing/sql_sales_transactions_day3.csv', format => 'csv', header => true)
WHERE modified_at > '2026-06-21 23:59:59'